In [1]:
import os
import torch

print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("torch.version.hip:", getattr(torch.version, "hip", None))
print("cuda_available:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
print("current_device:", torch.cuda.current_device() if torch.cuda.is_available() else None)
print("device_name:", torch.cuda.get_device_name(torch.cuda.current_device()) if torch.cuda.is_available() else None)
print("HIP_VISIBLE_DEVICES:", os.getenv("HIP_VISIBLE_DEVICES"))
print("ROCR_VISIBLE_DEVICES:", os.getenv("ROCR_VISIBLE_DEVICES"))
print("selected device var:", globals().get("device", "<not set yet>"))

torch: 2.9.1+rocm6.4
torch.version.cuda: None
torch.version.hip: 6.4.43484-123eb5128
cuda_available: True
device_count: 8
current_device: 0
device_name: AMD Instinct MI300X VF
HIP_VISIBLE_DEVICES: None
ROCR_VISIBLE_DEVICES: None
selected device var: <not set yet>


In [2]:
# ── Imports & config ─────────────────────────────────────────────────────────
import sys, gc, time, os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.warp import reproject, Resampling
from scipy import ndimage as ndi
from torchmetrics.functional.classification import binary_jaccard_index

# ── ROCm sanity checks (avoid silent CPU fallback) ───────────────────────────
_torch_hip = getattr(torch.version, 'hip', None)
if _torch_hip is None:
    raise RuntimeError(
        "Detected non-ROCm PyTorch build in this kernel. "
        "Install ROCm wheels, then restart the kernel:\n"
        "pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm6.4"
    )
if not torch.cuda.is_available() or torch.cuda.device_count() == 0:
    raise RuntimeError(
        "ROCm PyTorch is installed but no AMD GPU is visible to this kernel. "
        "Check kernel/env selection and HIP_VISIBLE_DEVICES/ROCR_VISIBLE_DEVICES."
    )

_ROOT = Path('/root/makeathon-challenge-2026')
sys.path.insert(0, str(_ROOT))
sys.path.insert(0, '/root/lucas')
from submission_utils import raster_to_geojson

# ── Multi-worker config (for 8-GPU parallel runs) ───────────────────────────
NUM_WORKERS = int(os.getenv('NUM_WORKERS', '1'))
WORKER_ID   = int(os.getenv('WORKER_ID', '0'))
GPU_ORDINAL = int(os.getenv('GPU_ORDINAL', os.getenv('LOCAL_RANK', '0')))
RUN_COORDINATOR = (WORKER_ID == 0)

# ── GPU setup ────────────────────────────────────────────────────────────────
n_visible = torch.cuda.device_count()
GPU_ORDINAL = GPU_ORDINAL % n_visible
torch.cuda.set_device(GPU_ORDINAL)
device = torch.device(f'cuda:{GPU_ORDINAL}')

print(f'Device : {device}')
print(f'  {torch.cuda.get_device_name(GPU_ORDINAL)}')
print(f'  VRAM  : {torch.cuda.get_device_properties(GPU_ORDINAL).total_memory/1e9:.1f} GB')
print(f'  Worker: {WORKER_ID}/{NUM_WORKERS}')
if hasattr(torch.backends, 'cudnn') and torch.backends.cudnn.is_available():
    torch.backends.cudnn.benchmark = True

# ── Paths ────────────────────────────────────────────────────────────────────
SLIM_DIR  = _ROOT / 'data' / 'cleaned' / 'features_slim'
LABEL_DIR = _ROOT / 'data' / 'cleaned' / 'fused_labels'
CKPT_DIR  = _ROOT / 'model_training' / 'multiseed_checkpoints'
PROB_DIR  = _ROOT / 'model_training' / 'multiseed_probs'
PRED_DIR  = _ROOT / 'model_training' / 'predictions_multiseed'
SUB_DIR   = _ROOT / 'model_training' / 'submissions_multiseed'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(PROB_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(SUB_DIR, exist_ok=True)

# ── Tile lists ───────────────────────────────────────────────────────────────
ALL_TRAIN = [
    '18NWG_6_6','18NWH_1_4','18NWJ_8_9','18NWM_9_4','18NXH_6_8','18NXJ_7_6',
    '18NYH_9_9','19NBD_4_4','47QMB_0_8','47QQV_2_4','48PUT_0_8','48PWV_7_8',
    '48PXC_7_7','48PYB_3_6','48QVE_3_0','48QWD_2_2',
]
VAL_TILES  = ['18NWG_6_6', '48PXC_7_7']
TRN_TILES  = [t for t in ALL_TRAIN if t not in VAL_TILES]
TEST_TILES = ['18NVJ_1_6','18NYH_2_1','33NTE_5_1','47QMA_6_2','48PWA_0_6']

# ── Training hyper-params ─────────────────────────────────────────────────────
# Reduced for ~15-minute total wall-clock target (6× parallel GPU workers)
PATCH_SIZE        = 256
N_EPOCHS          = 25
PATCHES_PER_EPOCH = 3200
BATCH_SIZE        = 128
LR                = 3e-4
BASE_CH           = 64

POS_WEIGHTS = [5.0, 5.0, 5.0, 3.5, 2.5, 7.0]
SEEDS       = list(range(len(POS_WEIGHTS)))
ASSIGNED_SEEDS = [s for s in SEEDS if s % max(NUM_WORKERS, 1) == WORKER_ID]

# ── Spatial post-processing params ───────────────────────────────────────────
MIN_BLOB_PX   = 50
FILL_HOLE_PX  = 200

print(f'Train tiles : {len(TRN_TILES)}, Val tiles: {len(VAL_TILES)}, Test: {len(TEST_TILES)}')
print(f'Patch size  : {PATCH_SIZE}px, Batch: {BATCH_SIZE}, base_ch={BASE_CH}')
print(f'N_EPOCHS={N_EPOCHS}, PATCHES_PER_EPOCH={PATCHES_PER_EPOCH} (steps/epoch={PATCHES_PER_EPOCH//BATCH_SIZE})')
print(f'All seeds   : {SEEDS}  pos_weights={POS_WEIGHTS}')
print(f'This worker seeds: {ASSIGNED_SEEDS}')
print(f'Coordinator mode: {RUN_COORDINATOR}')


Device : cuda:0
  AMD Instinct MI300X VF
  VRAM  : 205.8 GB
  Worker: 0/1
Train tiles : 14, Val tiles: 2, Test: 5
Patch size  : 256px, Batch: 128, base_ch=64
N_EPOCHS=25, PATCHES_PER_EPOCH=3200 (steps/epoch=25)
All seeds   : [0, 1, 2, 3, 4, 5]  pos_weights=[5.0, 5.0, 5.0, 3.5, 2.5, 7.0]
This worker seeds: [0, 1, 2, 3, 4, 5]
Coordinator mode: True


In [3]:
# True multi-GPU launcher (one process per GPU / worker)
import os
import sys
import shlex
import subprocess
from pathlib import Path

# Worker count: use at most visible GPUs and at most number of seeds
N = min(torch.cuda.device_count(), len(SEEDS))
NOTEBOOK_PATH = _ROOT / 'model_training' / 'multiseed_unet.ipynb'
LOG_DIR = _ROOT / 'model_training' / 'multiseed_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# IMPORTANT: In this environment, HIP/ROCR masking to values >0 hides all GPUs.
# So we do NOT set HIP_VISIBLE_DEVICES/ROCR_VISIBLE_DEVICES per worker.
# Instead, each worker gets all visible GPUs and selects its own via GPU_ORDINAL.
USE_HIP_MASK = False

print(f'Preparing true multi-GPU run with {N} workers for {len(SEEDS)} seeds...')
print(f'Notebook: {NOTEBOOK_PATH}')
print(f'Logs    : {LOG_DIR}')
print(f'USE_HIP_MASK={USE_HIP_MASK}')

commands = []
for wid in range(N):
    out_nb = _ROOT / 'model_training' / f'multiseed_unet_worker{wid}.ipynb'
    logf   = LOG_DIR / f'worker{wid}.log'

    env_vars = [
        f'NUM_WORKERS={N}',
        f'WORKER_ID={wid}',
        f'GPU_ORDINAL={wid}',
    ]
    if USE_HIP_MASK:
        env_vars += [f'HIP_VISIBLE_DEVICES={wid}', f'ROCR_VISIBLE_DEVICES={wid}']
    env_prefix = ' '.join(env_vars)

    # Use current kernel Python to avoid wrong Jupyter/PyTorch environment
    py = shlex.quote(sys.executable)
    nb = shlex.quote(str(NOTEBOOK_PATH))
    out_name = shlex.quote(out_nb.name)
    out_dir = shlex.quote(str(out_nb.parent))
    log_q = shlex.quote(str(logf))

    cmd = (
        f"{env_prefix} {py} -m jupyter nbconvert --to notebook --execute {nb} "
        f"--output {out_name} --output-dir {out_dir} > {log_q} 2>&1"
    )
    commands.append(cmd)
    print(f'[{wid}] {cmd}')

# Set True to launch from this cell.
LAUNCH_NOW = False

if LAUNCH_NOW:
    procs = []
    for wid, cmd in enumerate(commands):
        p = subprocess.Popen(cmd, shell=True, executable='/bin/bash')
        procs.append((wid, p.pid))
    print('\nLaunched workers:')
    for wid, pid in procs:
        print(f'  worker {wid} -> PID {pid}')
    print("\nMonitor with: rocm-smi --showpids --showuse --showmemuse")
    print(f"Logs in: {LOG_DIR}")
else:
    print('\nSet LAUNCH_NOW=True and re-run this cell to start all workers.')

Preparing true multi-GPU run with 6 workers for 6 seeds...
Notebook: /root/makeathon-challenge-2026/model_training/multiseed_unet.ipynb
Logs    : /root/makeathon-challenge-2026/model_training/multiseed_logs
USE_HIP_MASK=False
[0] NUM_WORKERS=6 WORKER_ID=0 GPU_ORDINAL=0 /root/makeathon-challenge-2026/venv/bin/python -m jupyter nbconvert --to notebook --execute /root/makeathon-challenge-2026/model_training/multiseed_unet.ipynb --output multiseed_unet_worker0.ipynb --output-dir /root/makeathon-challenge-2026/model_training > /root/makeathon-challenge-2026/model_training/multiseed_logs/worker0.log 2>&1
[1] NUM_WORKERS=6 WORKER_ID=1 GPU_ORDINAL=1 /root/makeathon-challenge-2026/venv/bin/python -m jupyter nbconvert --to notebook --execute /root/makeathon-challenge-2026/model_training/multiseed_unet.ipynb --output multiseed_unet_worker1.ipynb --output-dir /root/makeathon-challenge-2026/model_training > /root/makeathon-challenge-2026/model_training/multiseed_logs/worker1.log 2>&1
[2] NUM_WORKER

In [4]:
# ── Load ALL tiles into GPU VRAM ──────────────────────────────────────────────
# Features + labels live permanently on GPU. Zero CPU I/O during training.

def load_tile_cpu(tid, split='train'):
    """Load feature tif + reproject label onto feature grid. Returns numpy arrays."""
    feat_path = SLIM_DIR / split / f'{tid}_slim.tif'
    with rasterio.open(feat_path) as src:
        feat = src.read().astype(np.float32)          # (16, H, W)
        prof = src.profile.copy()
        f_transform, f_crs = src.transform, src.crs
        H, W = src.height, src.width
    np.nan_to_num(feat, copy=False, nan=0.0)

    lbl = None
    if split == 'train':
        lbl_path = LABEL_DIR / f'{tid}_fused.tif'
        lbl = np.full((H, W), 255, dtype=np.uint8)
        with rasterio.open(lbl_path) as src:
            raw = src.read(1)
            reproject(source=raw, destination=lbl,
                      src_transform=src.transform, src_crs=src.crs,
                      dst_transform=f_transform,   dst_crs=f_crs,
                      resampling=Resampling.nearest)
    return feat, lbl, prof

print('Loading tiles into CPU RAM...')
cpu_data = {}   # tid → (feat_np, lbl_np, prof)
for tid in ALL_TRAIN:
    feat, lbl, prof = load_tile_cpu(tid, 'train')
    cpu_data[tid] = (feat, lbl, prof)

# Compute normalisation stats from training tiles only
band_means = np.zeros(16, dtype=np.float32)
band_stds  = np.ones(16,  dtype=np.float32)
for b in range(16):
    vals = np.concatenate([cpu_data[t][0][b].ravel() for t in TRN_TILES])
    band_means[b] = vals.mean()
    band_stds[b]  = vals.std() + 1e-8

# Normalise and push everything to GPU
print('Pushing tensors to GPU VRAM...')
gpu_feats = {}   # tid → (16, H, W) float32 CUDA tensor
gpu_labels = {}  # tid → (H, W) uint8 CUDA tensor
gpu_profs  = {}  # tid → rasterio profile

for tid in ALL_TRAIN:
    feat_np, lbl_np, prof = cpu_data[tid]
    feat_norm = feat_np.copy()
    for b in range(16):
        feat_norm[b] = (feat_norm[b] - band_means[b]) / band_stds[b]
    gpu_feats[tid]  = torch.from_numpy(feat_norm).to(device)
    gpu_labels[tid] = torch.from_numpy(lbl_np.copy()).to(device)
    gpu_profs[tid]  = prof

vram_used = torch.cuda.memory_allocated() / 1e9
print(f'All train tiles on GPU. VRAM used: {vram_used:.2f} GB')

# Precompute positive pixel locations on CPU for oversampling
pos_locs = {}   # tid → (rows, cols) numpy arrays
for tid in TRN_TILES:
    lbl_np = cpu_data[tid][1]
    rows, cols = np.where(lbl_np == 1)
    if len(rows) > 0:
        pos_locs[tid] = (rows, cols)
        
del cpu_data
gc.collect()
print(f'Positive tiles in train set: {len(pos_locs)}/{len(TRN_TILES)}')

Loading tiles into CPU RAM...
Pushing tensors to GPU VRAM...
All train tiles on GPU. VRAM used: 1.06 GB
Positive tiles in train set: 14/14


In [5]:
# ── Model architecture ────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=16, base=32):
        super().__init__()
        ch = [base, base*2, base*4, base*8]
        self.enc1 = ConvBlock(in_channels, ch[0])
        self.enc2 = ConvBlock(ch[0], ch[1])
        self.enc3 = ConvBlock(ch[1], ch[2])
        self.enc4 = ConvBlock(ch[2], ch[3])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(ch[3], ch[3]*2)
        self.up4  = nn.ConvTranspose2d(ch[3]*2, ch[3], 2, stride=2)
        self.dec4 = ConvBlock(ch[3]*2, ch[3])
        self.up3  = nn.ConvTranspose2d(ch[3],   ch[2], 2, stride=2)
        self.dec3 = ConvBlock(ch[2]*2, ch[2])
        self.up2  = nn.ConvTranspose2d(ch[2],   ch[1], 2, stride=2)
        self.dec2 = ConvBlock(ch[1]*2, ch[1])
        self.up1  = nn.ConvTranspose2d(ch[1],   ch[0], 2, stride=2)
        self.dec1 = ConvBlock(ch[0]*2, ch[0])
        self.out_conv = nn.Conv2d(ch[0], 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)

class BCEDiceLoss(nn.Module):
    def __init__(self, pos_weight, dice_weight=0.5):
        super().__init__()
        self.pos_weight  = torch.tensor([pos_weight], device=device)
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        # ignore index 255 (unlabelled / small-patch)
        valid    = targets < 255
        logits_v = logits[valid]
        targets_v = targets[valid].float()
        weight = torch.where(targets_v == 1, self.pos_weight, torch.ones_like(targets_v))
        bce = F.binary_cross_entropy_with_logits(logits_v, targets_v, weight=weight)
        probs = torch.sigmoid(logits_v)
        inter = (probs * targets_v).sum()
        dice  = 1 - (2*inter + 1) / (probs.sum() + targets_v.sum() + 1)
        return bce + self.dice_weight * dice

print('Model architecture defined.')

Model architecture defined.


In [6]:
# ── GPU-resident patch sampler ────────────────────────────────────────────────
# All slicing happens on GPU tensors — zero CPU overhead during training.

def sample_batch_gpu(tile_ids, gpu_feats, gpu_labels, pos_locs,
                     ps=PATCH_SIZE, batch_size=BATCH_SIZE, pos_oversample=3):
    """
    Sample a batch of patches entirely on GPU.
    pos_oversample: how many times more likely to pick a positive-containing patch.
    Returns:
        patches : (B, 16, ps, ps)  float32 GPU tensor
        labels  : (B, ps, ps)      uint8  GPU tensor
    """
    patches_list = []
    labels_list  = []
    pos_tiles = [t for t in tile_ids if t in pos_locs]

    for _ in range(batch_size):
        use_pos = (np.random.rand() < pos_oversample / (1 + pos_oversample)
                   and len(pos_tiles) > 0)
        if use_pos:
            tid  = pos_tiles[np.random.randint(len(pos_tiles))]
            rows, cols = pos_locs[tid]
            idx  = np.random.randint(len(rows))
            cr   = int(np.clip(rows[idx] - ps//2, 0, gpu_feats[tid].shape[1] - ps))
            cc   = int(np.clip(cols[idx] - ps//2, 0, gpu_feats[tid].shape[2] - ps))
        else:
            tid  = tile_ids[np.random.randint(len(tile_ids))]
            cr   = np.random.randint(0, max(1, gpu_feats[tid].shape[1] - ps))
            cc   = np.random.randint(0, max(1, gpu_feats[tid].shape[2] - ps))

        patches_list.append(gpu_feats[tid][:, cr:cr+ps, cc:cc+ps])
        labels_list.append(gpu_labels[tid][cr:cr+ps, cc:cc+ps])

    patches = torch.stack(patches_list)   # (B, 16, ps, ps)
    labels  = torch.stack(labels_list)    # (B, ps, ps)

    # Random flips + 90° rotation augmentation — on GPU
    if np.random.rand() > 0.5:
        patches = patches.flip(-1); labels = labels.flip(-1)
    if np.random.rand() > 0.5:
        patches = patches.flip(-2); labels = labels.flip(-2)
    k = np.random.randint(4)
    if k > 0:
        patches = torch.rot90(patches, k, [-2, -1])
        labels  = torch.rot90(labels,  k, [-2, -1])

    return patches, labels

print('GPU patch sampler defined.')

GPU patch sampler defined.


In [7]:
# ── Full-tile inference (TTA: orig + hflip + vflip) ──────────────────────────
# Input tensor already lives on GPU — no transfers during inference.

@torch.inference_mode()
def infer_tile_tta(feat_gpu, model, ps=PATCH_SIZE, stride_factor=2, batch_size=512):
    """
    feat_gpu : (16, H, W) float32 GPU tensor (already normalised).
    Returns  : (H, W) float32 CPU numpy array of probabilities.
    """
    C, H, W = feat_gpu.shape
    stride = ps // stride_factor

    pad_h = (ps - H % ps) % ps
    pad_w = (ps - W % ps) % ps
    fp = F.pad(feat_gpu, (0, pad_w, 0, pad_h), mode='reflect')   # still on GPU
    Hp, Wp = fp.shape[1], fp.shape[2]

    prob_sum  = torch.zeros(Hp, Wp, device=device)
    count_map = torch.zeros(Hp, Wp, device=device)

    positions = [(r, c)
                 for r in range(0, Hp - ps + 1, stride)
                 for c in range(0, Wp - ps + 1, stride)]

    for i in range(0, len(positions), batch_size):
        batch_pos = positions[i : i + batch_size]
        t = torch.stack([fp[:, r:r+ps, c:c+ps] for r, c in batch_pos])  # (B,16,ps,ps)

        p0 = torch.sigmoid(model(t)).squeeze(1)
        p1 = torch.sigmoid(model(t.flip(-1))).squeeze(1).flip(-1)
        p2 = torch.sigmoid(model(t.flip(-2))).squeeze(1).flip(-2)
        probs = (p0 + p1 + p2) / 3.0   # (B, ps, ps) — stays on GPU

        for k, (r, c) in enumerate(batch_pos):
            prob_sum[r:r+ps, c:c+ps]  += probs[k]
            count_map[r:r+ps, c:c+ps] += 1.0

    return (prob_sum / count_map.clamp(min=1))[:H, :W].cpu().numpy()

print('TTA inference function defined.')

TTA inference function defined.


In [8]:

# ── Spatial post-processing ───────────────────────────────────────────────────
# Applied AFTER thresholding. Targets two cheap IoU wins:
#   1. Remove tiny FP blobs  — they add to union without adding to intersection
#   2. Fill small FN holes   — interior gaps inside real defor patches are likely FN

def postprocess_pred(pred_binary, min_blob_px=MIN_BLOB_PX, fill_hole_px=FILL_HOLE_PX):
    """
    pred_binary : (H, W) uint8/int array with values {0, 1}.
    Returns     : (H, W) uint8 cleaned array.
    """
    mask = pred_binary.astype(bool)

    # ── 1. Remove small FP blobs ─────────────────────────────────────────────
    labeled, n = ndi.label(mask)
    if n > 0:
        sizes = ndi.sum(mask, labeled, range(1, n + 1))
        small = np.array(sizes) < min_blob_px
        remove_mask = small[labeled - 1]        # maps label → bool
        remove_mask[labeled == 0] = False        # background stays
        mask[remove_mask] = False

    # ── 2. Fill small FN holes inside predicted regions ──────────────────────
    inverted  = ~mask
    labeled_holes, n_holes = ndi.label(inverted)
    if n_holes > 0:
        hole_sizes = ndi.sum(inverted, labeled_holes, range(1, n_holes + 1))
        # Only fill holes that don't touch the image border
        border = np.zeros_like(inverted)
        border[0, :] = border[-1, :] = border[:, 0] = border[:, -1] = True
        border_labels = set(labeled_holes[border].tolist()) - {0}
        small_interior = [
            i for i, sz in enumerate(hole_sizes, start=1)
            if sz < fill_hole_px and i not in border_labels
        ]
        for lbl in small_interior:
            mask[labeled_holes == lbl] = True

    return mask.astype(np.uint8)

print(f'Post-processing defined: remove blobs <{MIN_BLOB_PX}px, fill holes <{FILL_HOLE_PX}px.')


Post-processing defined: remove blobs <50px, fill holes <200px.


In [9]:
# ── Train all seeds ───────────────────────────────────────────────────────────
# Parallel mode: run the notebook once per GPU with different
#   NUM_WORKERS / WORKER_ID / GPU_ORDINAL environment variables.

def seed_probs_exist(seed):
    return all((PROB_DIR / f'seed{seed}_{tid}.npy').exists() for tid in ALL_TRAIN)

def train_seed(seed, pos_weight):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = UNet(in_channels=16, base=BASE_CH).to(device)
    criterion = BCEDiceLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-6)

    steps_per_epoch = PATCHES_PER_EPOCH // BATCH_SIZE
    best_val_iou = -1.0
    best_state   = None

    t0 = time.time()
    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for _ in range(steps_per_epoch):
            patches, labels = sample_batch_gpu(
                TRN_TILES, gpu_feats, gpu_labels, pos_locs, batch_size=BATCH_SIZE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(patches).squeeze(1)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()

        if epoch % 5 == 0 or epoch == N_EPOCHS:
            model.eval()
            val_ious = []
            for vtid in VAL_TILES:
                prob = infer_tile_tta(gpu_feats[vtid], model)
                lbl  = gpu_labels[vtid].cpu().numpy()
                pred = (prob > 0.5).astype(np.float32)
                valid = lbl < 255
                iou = float(binary_jaccard_index(
                    torch.from_numpy(pred[valid]),
                    torch.from_numpy(lbl[valid].astype(np.int64)),
                    threshold=0.5, zero_division=1.0))
                val_ious.append(iou)
            mean_val = float(np.mean(val_ious))
            elapsed  = time.time() - t0
            print(f'  Seed {seed} | Ep {epoch:3d}/{N_EPOCHS} '
                  f'| loss={epoch_loss/steps_per_epoch:.4f} '
                  f'| val_IoU={mean_val:.4f} '
                  f'| {elapsed:.0f}s')
            if mean_val > best_val_iou:
                best_val_iou = mean_val
                best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    ckpt_path = CKPT_DIR / f'unet_seed{seed}.pt'
    torch.save(best_state, ckpt_path)
    print(f'Seed {seed}: best val IoU={best_val_iou:.4f}  saved → {ckpt_path.name}')

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    for tid in ALL_TRAIN:
        prob = infer_tile_tta(gpu_feats[tid], model)
        np.save(PROB_DIR / f'seed{seed}_{tid}.npy', prob)

    del model, criterion, optimizer, scheduler
    torch.cuda.empty_cache()
    return best_val_iou


# ── Run assigned seeds only (parallel-safe) ──────────────────────────────────
seed_results = {}
for s in ASSIGNED_SEEDS:
    pw = POS_WEIGHTS[s]
    ckpt_exists = (CKPT_DIR / f'unet_seed{s}.pt').exists()
    probs_exist = seed_probs_exist(s)

    if ckpt_exists and probs_exist:
        print(f'\n══ Skip seed {s}: checkpoint + prob maps already exist ══')
        continue

    print(f'\n══ Training seed {s}  pos_weight={pw}  worker={WORKER_ID}/{NUM_WORKERS} ══')
    val_iou = train_seed(s, pw)
    seed_results[s] = val_iou

print('\n── Worker summary ──')
if len(seed_results) == 0:
    print('No seeds trained in this worker run (all assigned seeds already finished).')
else:
    for s, iou in seed_results.items():
        print(f'  Seed {s} (pw={POS_WEIGHTS[s]}): best val IoU = {iou:.4f}')



══ Training seed 0  pos_weight=5.0  worker=0/1 ══


  Seed 0 | Ep   5/25 | loss=0.6476 | val_IoU=0.5484 | 1558s
  Seed 0 | Ep  10/25 | loss=0.5463 | val_IoU=0.5170 | 1711s
  Seed 0 | Ep  15/25 | loss=0.4829 | val_IoU=0.5363 | 1865s
  Seed 0 | Ep  20/25 | loss=0.4539 | val_IoU=0.5302 | 2018s
  Seed 0 | Ep  25/25 | loss=0.4391 | val_IoU=0.5306 | 2149s
Seed 0: best val IoU=0.5484  saved → unet_seed0.pt

══ Skip seed 1: checkpoint + prob maps already exist ══

══ Skip seed 2: checkpoint + prob maps already exist ══

══ Skip seed 3: checkpoint + prob maps already exist ══

══ Skip seed 4: checkpoint + prob maps already exist ══

══ Skip seed 5: checkpoint + prob maps already exist ══

── Worker summary ──
  Seed 0 (pw=5.0): best val IoU = 0.5484


In [10]:
# ── Ensemble: average prob maps across finished seeds ─────────────────────────

def seed_ready(seed):
    ckpt_ok = (CKPT_DIR / f'unet_seed{seed}.pt').exists()
    probs_ok = all((PROB_DIR / f'seed{seed}_{tid}.npy').exists() for tid in ALL_TRAIN)
    return ckpt_ok and probs_ok

if not RUN_COORDINATOR:
    print(f'Worker {WORKER_ID}: skipping ensemble/calibration/inference. Worker 0 handles this.')
else:
    print('Worker 0 waiting for all seeds to be ready...')
    while not all(seed_ready(s) for s in SEEDS):
        ready = [s for s in SEEDS if seed_ready(s)]
        print(f'  ready={ready}/{SEEDS} ... sleeping 30s')
        time.sleep(30)

    READY_SEEDS = [s for s in SEEDS if seed_ready(s)]
    if len(READY_SEEDS) == 0:
        raise RuntimeError('No completed seeds found. Train at least one seed first.')

    print(f'Ready seeds for ensembling: {READY_SEEDS}')
    ens_train_probs = {}

    for tid in ALL_TRAIN:
        seed_probs = [np.load(PROB_DIR / f'seed{s}_{tid}.npy') for s in READY_SEEDS]
        ens_train_probs[tid] = np.mean(seed_probs, axis=0)

    print(f'Ensemble probs computed for {len(ens_train_probs)} tiles from {len(READY_SEEDS)} seeds.')
    for tid, p in ens_train_probs.items():
        print(f'  {tid}: shape={p.shape}  mean={p.mean():.3f}  max={p.max():.3f}')

Worker 0 waiting for all seeds to be ready...
Ready seeds for ensembling: [0, 1, 2, 3, 4, 5]
Ensemble probs computed for 16 tiles from 6 seeds.
  18NWG_6_6: shape=(1004, 998)  mean=0.423  max=1.000
  18NWH_1_4: shape=(1004, 999)  mean=0.116  max=0.997
  18NWJ_8_9: shape=(1004, 999)  mean=0.148  max=0.996
  18NWM_9_4: shape=(1003, 1002)  mean=0.154  max=0.998
  18NXH_6_8: shape=(1005, 999)  mean=0.337  max=1.000
  18NXJ_7_6: shape=(1005, 1000)  mean=0.125  max=0.993
  18NYH_9_9: shape=(1006, 999)  mean=0.263  max=1.000
  19NBD_4_4: shape=(1006, 1001)  mean=0.155  max=0.997
  47QMB_0_8: shape=(980, 1031)  mean=0.104  max=0.954
  47QQV_2_4: shape=(991, 1034)  mean=0.109  max=0.942
  48PUT_0_8: shape=(1001, 1013)  mean=0.195  max=1.000
  48PWV_7_8: shape=(994, 1014)  mean=0.600  max=1.000
  48PXC_7_7: shape=(993, 1025)  mean=0.207  max=1.000
  48PYB_3_6: shape=(997, 1025)  mean=0.320  max=1.000
  48QVE_3_0: shape=(982, 1026)  mean=0.314  max=1.000
  48QWD_2_2: shape=(983, 1021)  mean=0.325

In [11]:
# ── Threshold calibration on VAL tiles only (honest) ─────────────────────────
if not RUN_COORDINATOR:
    print(f'Worker {WORKER_ID}: skipping threshold calibration.')
else:
    val_labels = {tid: gpu_labels[tid].cpu().numpy() for tid in VAL_TILES}

    def iou_at_threshold(prob_maps, label_dict, tile_list, threshold):
        ious = []
        for tid in tile_list:
            prob = prob_maps[tid]
            lbl  = label_dict[tid]
            pred  = (prob > threshold).astype(np.float32)
            valid = lbl < 255
            iou = float(binary_jaccard_index(
                torch.from_numpy(pred[valid]),
                torch.from_numpy(lbl[valid].astype(np.int64)),
                threshold=0.5, zero_division=1.0))
            ious.append(iou)
        return float(np.mean(ious))

    thresholds = np.arange(0.20, 0.85, 0.01)
    val_scores = [iou_at_threshold(ens_train_probs, val_labels, VAL_TILES, t)
                  for t in thresholds]

    best_idx   = int(np.argmax(val_scores))
    BEST_THRESH = float(thresholds[best_idx])

    print(f'Val-calibrated threshold : {BEST_THRESH:.2f}')
    print(f'Val IoU at best threshold: {val_scores[best_idx]:.4f}')
    print(f'Val IoU at 0.50          : {iou_at_threshold(ens_train_probs, val_labels, VAL_TILES, 0.5):.4f}')

    all_labels = {tid: gpu_labels[tid].cpu().numpy() for tid in ALL_TRAIN}
    all_score  = iou_at_threshold(ens_train_probs, all_labels, ALL_TRAIN, BEST_THRESH)
    print(f'\nAll-train IoU @ {BEST_THRESH:.2f}: {all_score:.4f}  (in-sample — for reference only)')

Val-calibrated threshold : 0.51
Val IoU at best threshold: 0.5523
Val IoU at 0.50          : 0.5522

All-train IoU @ 0.51: 0.5247  (in-sample — for reference only)


In [12]:
# ── Per-tile IoU report on all training tiles ─────────────────────────────────
if not RUN_COORDINATOR:
    print(f'Worker {WORKER_ID}: skipping per-tile IoU report.')
else:
    print(f'\n── Multi-seed ensemble (threshold={BEST_THRESH:.2f}) ──')
    ious = []
    for tid in ALL_TRAIN:
        prob  = ens_train_probs[tid]
        lbl   = all_labels[tid]
        pred  = (prob > BEST_THRESH).astype(np.float32)
        valid = lbl < 255
        iou = float(binary_jaccard_index(
            torch.from_numpy(pred[valid]),
            torch.from_numpy(lbl[valid].astype(np.int64)),
            threshold=0.5, zero_division=1.0))
        ious.append(iou)
        tag = ' ← val' if tid in VAL_TILES else ''
        print(f'  {tid:<18} IoU={iou:.4f}{tag}')

    print(f'\n  Mean : {np.mean(ious):.4f} ± {np.std(ious):.4f}')
    print(f'  Min  : {np.min(ious):.4f}   Max: {np.max(ious):.4f}')
    val_ious = [ious[ALL_TRAIN.index(t)] for t in VAL_TILES]
    print(f'  Val-only mean: {np.mean(val_ious):.4f}  (honest estimate)')


── Multi-seed ensemble (threshold=0.51) ──
  18NWG_6_6          IoU=0.7082 ← val
  18NWH_1_4          IoU=0.3869
  18NWJ_8_9          IoU=0.4609
  18NWM_9_4          IoU=0.4025
  18NXH_6_8          IoU=0.7455
  18NXJ_7_6          IoU=0.4058
  18NYH_9_9          IoU=0.6949
  19NBD_4_4          IoU=0.5454
  47QMB_0_8          IoU=0.3097
  47QQV_2_4          IoU=0.2594
  48PUT_0_8          IoU=0.5173
  48PWV_7_8          IoU=0.7528
  48PXC_7_7          IoU=0.3964 ← val
  48PYB_3_6          IoU=0.6881
  48QVE_3_0          IoU=0.5857
  48QWD_2_2          IoU=0.5356

  Mean : 0.5247 ± 0.1540
  Min  : 0.2594   Max: 0.7528
  Val-only mean: 0.5523  (honest estimate)


In [13]:
# ── Generate test predictions ─────────────────────────────────────────────────
if not RUN_COORDINATOR:
    print(f'Worker {WORKER_ID}: skipping test prediction export.')
else:
    print('Generating test predictions...')
    t0 = time.time()

    if 'READY_SEEDS' not in globals() or len(READY_SEEDS) == 0:
        READY_SEEDS = [s for s in SEEDS if (CKPT_DIR / f'unet_seed{s}.pt').exists()]
    if len(READY_SEEDS) == 0:
        raise RuntimeError('No seed checkpoints found. Train seeds first.')

    models = []
    for s in READY_SEEDS:
        m = UNet(in_channels=16, base=BASE_CH).to(device)
        state = torch.load(CKPT_DIR / f'unet_seed{s}.pt', map_location=device)
        m.load_state_dict(state)
        m.eval()
        models.append(m)
    print(f'Loaded {len(models)} seed checkpoints: {READY_SEEDS}')

    for tid in TEST_TILES:
        feat_np, _, prof = load_tile_cpu(tid, 'test')
        feat_norm = feat_np.copy()
        for b in range(16):
            feat_norm[b] = (feat_norm[b] - band_means[b]) / band_stds[b]
        feat_gpu = torch.from_numpy(feat_norm).to(device)
        H, W = feat_gpu.shape[1], feat_gpu.shape[2]

        seed_probs = [infer_tile_tta(feat_gpu, m) for m in models]
        ens_prob = np.mean(seed_probs, axis=0)

        pred_binary = postprocess_pred((ens_prob > BEST_THRESH).astype(np.uint8))

        out_path = PRED_DIR / f'{tid}_pred_multiseed.tif'
        out_prof  = prof.copy()
        out_prof.update(count=1, dtype='uint8', compress='deflate', nodata=0)
        for k in ['blockxsize', 'blockysize', 'tiled']:
            out_prof.pop(k, None)
        with rasterio.open(out_path, 'w', **out_prof) as dst:
            dst.write(pred_binary, 1)

        n_defor = pred_binary.sum()
        print(f'  {tid}: {H}x{W}  defor={n_defor:,} ({100*n_defor/(H*W):.1f}%)  {time.time()-t0:.1f}s')

    if 'feat_gpu' in locals():
        del feat_gpu
    torch.cuda.empty_cache()
    print(f'\nAll test predictions saved to {PRED_DIR}')

Generating test predictions...
Loaded 6 seed checkpoints: [0, 1, 2, 3, 4, 5]
  18NVJ_1_6: 1004x999  defor=0 (0.0%)  2.7s
  18NYH_2_1: 1005x1000  defor=88,646 (8.8%)  4.3s
  33NTE_5_1: 1006x1002  defor=71,469 (7.1%)  5.9s
  47QMA_6_2: 978x1028  defor=25,612 (2.5%)  8.0s
  48PWA_0_6: 989x1013  defor=185,885 (18.6%)  9.6s

All test predictions saved to /root/makeathon-challenge-2026/model_training/predictions_multiseed


In [14]:
# ── Official benchmark report ─────────────────────────────────────────────────
if not RUN_COORDINATOR:
    print(f'Worker {WORKER_ID}: skipping benchmark report.')
else:
    if 'READY_SEEDS' not in globals():
        READY_SEEDS = [s for s in SEEDS if (CKPT_DIR / f'unet_seed{s}.pt').exists()]

    print('\n══════════════════════════════════════════════════════════════')
    print(f'  Multi-seed ensemble  |  seeds={len(READY_SEEDS)} {READY_SEEDS}  |  base={BASE_CH}')
    print(f'  threshold={BEST_THRESH:.2f}  |  postproc=True')
    print('══════════════════════════════════════════════════════════════')
    print(f'  {"Tile":<18}  {"IoU":>6}  {"Split"}')
    print('  ──────────────────────────────────────────────────────────')

    all_ious, val_ious_list, trn_ious_list = [], [], []
    for tid in ALL_TRAIN:
        prob  = ens_train_probs[tid]
        lbl   = all_labels[tid]
        pred  = postprocess_pred((prob > BEST_THRESH).astype(np.uint8)).astype(np.float32)
        valid = lbl < 255
        iou = float(binary_jaccard_index(
            torch.from_numpy(pred[valid]),
            torch.from_numpy(lbl[valid].astype(np.int64)),
            threshold=0.5, zero_division=1.0))
        all_ious.append(iou)
        split = 'VAL ←' if tid in VAL_TILES else 'train'
        if tid in VAL_TILES:
            val_ious_list.append(iou)
        else:
            trn_ious_list.append(iou)
        print(f'  {tid:<18}  {iou:>6.4f}  {split}')

    print('  ──────────────────────────────────────────────────────────')
    print(f'  {"All 16 tiles":<18}  {np.mean(all_ious):>6.4f}  mean ± {np.std(all_ious):.4f}')
    print(f'  {"Val only (honest)":<18}  {np.mean(val_ious_list):>6.4f}  ← competition proxy')
    print(f'  {"Train only":<18}  {np.mean(trn_ious_list):>6.4f}  (in-sample, inflated)')
    print('══════════════════════════════════════════════════════════════')


══════════════════════════════════════════════════════════════
  Multi-seed ensemble  |  seeds=6 [0, 1, 2, 3, 4, 5]  |  base=64
  threshold=0.51  |  postproc=True
══════════════════════════════════════════════════════════════
  Tile                   IoU  Split
  ──────────────────────────────────────────────────────────
  18NWG_6_6           0.7082  VAL ←
  18NWH_1_4           0.3871  train
  18NWJ_8_9           0.4627  train
  18NWM_9_4           0.4021  train
  18NXH_6_8           0.7443  train
  18NXJ_7_6           0.4120  train
  18NYH_9_9           0.6939  train
  19NBD_4_4           0.5443  train
  47QMB_0_8           0.3151  train
  47QQV_2_4           0.2616  train
  48PUT_0_8           0.5184  train
  48PWV_7_8           0.7487  train
  48PXC_7_7           0.3932  VAL ←
  48PYB_3_6           0.6857  train
  48QVE_3_0           0.5831  train
  48QWD_2_2           0.5345  train
  ──────────────────────────────────────────────────────────
  All 16 tiles        0.5247  mean ± 0.

In [15]:
# ── Generate submission GeoJSONs ──────────────────────────────────────────────
if not RUN_COORDINATOR:
    print(f'Worker {WORKER_ID}: skipping submission GeoJSON export.')
else:
    for tid in TEST_TILES:
        pred_path = PRED_DIR / f'{tid}_pred_multiseed.tif'
        out_path  = SUB_DIR  / f'{tid}_submission.geojson'
        try:
            gj = raster_to_geojson(pred_path, output_path=out_path, min_area_ha=0.5)
            print(f'  {tid}: {len(gj["features"])} polygons → {out_path.name}')
        except ValueError as e:
            print(f'  {tid}: ⚠ {e}')

    print(f'\nSubmissions saved to {SUB_DIR}')

  18NVJ_1_6: ⚠ No deforestation pixels (value=1) found in /root/makeathon-challenge-2026/model_training/predictions_multiseed/18NVJ_1_6_pred_multiseed.tif. Ensure the raster has been binarised before calling this function.
  18NYH_2_1: 161 polygons → 18NYH_2_1_submission.geojson
  33NTE_5_1: 216 polygons → 33NTE_5_1_submission.geojson
  47QMA_6_2: 167 polygons → 47QMA_6_2_submission.geojson
  48PWA_0_6: 360 polygons → 48PWA_0_6_submission.geojson

Submissions saved to /root/makeathon-challenge-2026/model_training/submissions_multiseed


In [16]:
# ── Save run IoU + notebook artifacts to one results file ────────────────────
import json
from datetime import datetime

if not RUN_COORDINATOR:
    print(f'Worker {WORKER_ID}: skipping results export (worker 0 handles this).')
else:
    RESULTS_DIR = _ROOT / 'results'
    os.makedirs(RESULTS_DIR, exist_ok=True)

    required = ['ens_train_probs', 'all_labels', 'BEST_THRESH']
    missing = [k for k in required if k not in globals()]
    if len(missing) > 0:
        raise RuntimeError(f'Missing required variables for IoU/results export: {missing}. Run cells 9-13 first.')

    # Recompute per-tile IoU with final post-processing (same as benchmark logic)
    per_tile = []
    for tid in ALL_TRAIN:
        prob = ens_train_probs[tid]
        lbl  = all_labels[tid]
        pred = postprocess_pred((prob > BEST_THRESH).astype(np.uint8)).astype(np.float32)
        valid = lbl < 255
        iou = float(binary_jaccard_index(
            torch.from_numpy(pred[valid]),
            torch.from_numpy(lbl[valid].astype(np.int64)),
            threshold=0.5, zero_division=1.0))
        split = 'val' if tid in VAL_TILES else 'train'
        per_tile.append({'tile_id': tid, 'split': split, 'iou': iou})

    all_ious = [r['iou'] for r in per_tile]
    val_ious = [r['iou'] for r in per_tile if r['split'] == 'val']
    trn_ious = [r['iou'] for r in per_tile if r['split'] == 'train']

    summary = {
        'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
        'threshold': float(BEST_THRESH),
        'num_ready_seeds': int(len(READY_SEEDS)) if 'READY_SEEDS' in globals() else 0,
        'ready_seeds': [int(s) for s in READY_SEEDS] if 'READY_SEEDS' in globals() else [],
        'all_tiles_mean_iou': float(np.mean(all_ious)),
        'all_tiles_std_iou': float(np.std(all_ious)),
        'val_mean_iou': float(np.mean(val_ious)) if len(val_ious) else None,
        'train_mean_iou': float(np.mean(trn_ious)) if len(trn_ious) else None,
        'n_epochs': int(N_EPOCHS),
        'patch_size': int(PATCH_SIZE),
        'batch_size': int(BATCH_SIZE),
        'base_ch': int(BASE_CH),
        'lr': float(LR),
        'pos_weights': [float(x) for x in POS_WEIGHTS],
        'val_tiles': VAL_TILES,
        'test_tiles': TEST_TILES,
        'pred_dir': str(PRED_DIR),
        'sub_dir': str(SUB_DIR),
    }

    metadata = {
        'worker_id': int(WORKER_ID),
        'num_workers': int(NUM_WORKERS),
        'gpu_ordinal': int(GPU_ORDINAL),
        'device': str(device),
        'assigned_seeds': [int(s) for s in ASSIGNED_SEEDS],
        'all_seeds': [int(s) for s in SEEDS],
        'checkpoint_dir': str(CKPT_DIR),
        'prob_dir': str(PROB_DIR),
        'results_dir': str(RESULTS_DIR),
    }

    threshold_scan = None
    if 'thresholds' in globals() and 'val_scores' in globals():
        threshold_scan = {
            'thresholds': [float(t) for t in thresholds],
            'val_scores': [float(s) for s in val_scores],
            'best_threshold': float(BEST_THRESH),
        }

    artifacts_manifest = {
        'checkpoints': sorted([p.name for p in CKPT_DIR.glob('unet_seed*.pt')]),
        'prob_maps_count': len(list(PROB_DIR.glob('seed*_*.npy'))),
        'predictions': sorted([p.name for p in PRED_DIR.glob('*_pred_multiseed.tif')]),
        'submissions': sorted([p.name for p in SUB_DIR.glob('*_submission.geojson')]),
    }

    # Single-file export
    results_blob = {
        'summary': summary,
        'metadata': metadata,
        'threshold_scan': threshold_scan,
        'per_tile_iou': per_tile,
        'artifacts_manifest': artifacts_manifest,
    }

    out_file = RESULTS_DIR / 'run_results.json'
    with open(out_file, 'w') as f:
        json.dump(results_blob, f, indent=2)

    print('\nSaved single results file:')
    print(f'  - {out_file}')
    print(f"\nVal mean IoU: {summary['val_mean_iou']:.4f} | All mean IoU: {summary['all_tiles_mean_iou']:.4f}")


Saved single results file:
  - /root/makeathon-challenge-2026/results/run_results.json

Val mean IoU: 0.5507 | All mean IoU: 0.5247


/tmp/ipykernel_96954/391769013.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
